In [1]:
import tensorflow as tf
from keras import layers
import pandas as pd
import numpy as np

@tf.keras.utils.register_keras_serializable()
class AnomalyDetector(tf.keras.Model):
    def __init__(self, **kwargs):
        super(AnomalyDetector, self).__init__(**kwargs)
        self.encoder = tf.keras.Sequential([
          # layers.Dense(16, activation="relu"),
          # # Model was overfitting so add dropout layers
          # layers.Dropout(0.2),
          # layers.Dense(8, activation="relu"),
          # layers.Dropout(0.2),
          # layers.Dense(4, activation="relu")

            layers.Dense(32, activation=None),
            layers.BatchNormalization(),
            layers.LeakyReLU(),
            layers.Dropout(0.1),
            layers.Dense(16, activation=None),
            layers.BatchNormalization(),
            layers.LeakyReLU(),
            layers.Dense(8, activation=None),
            layers.BatchNormalization(),
            layers.LeakyReLU()
        ])
    
        self.decoder = tf.keras.Sequential([

            layers.Dense(16, activation=None),
            layers.BatchNormalization(),
            layers.LeakyReLU(),
            layers.Dropout(0.1),
            layers.Dense(32, activation=None),
            layers.BatchNormalization(),
            layers.LeakyReLU(),
            layers.Dense(24)
            
          # layers.Dense(8, activation="relu"),
          # layers.Dropout(0.2),
          # layers.Dense(16, activation="relu"),
          # layers.Dropout(0.1),
          # layers.Dense(24)
        ])
      
    def call(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

    def get_config(self):
        # Return config dictionary; empty because no constructor args
        base_config = super(AnomalyDetector, self).get_config()
        return base_config

    @classmethod
    def from_config(cls, config):
        return cls(**config)

2025-07-24 10:58:04.114647: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-24 10:58:04.194608: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
cae_model = tf.keras.models.load_model(
    "Models/8f_cae_model.keras",
    custom_objects={"AnomalyDetector": AnomalyDetector}
)

In [3]:
transformed_training_data = pd.read_csv('Dataset/Transformed/Normalised/labelled_transformed_training_data.csv')

# 1. Extract labels (column 6)
training_labels = transformed_training_data.iloc[:, 6].astype(bool)  # Anomalies: True

# 2. Drop label and extra column (column 6 and 7)
training_data = transformed_training_data.drop(
    columns=[transformed_training_data.columns[6], transformed_training_data.columns[7]]
)

# 3. Convert to float32 tensor
training_data = tf.convert_to_tensor(training_data.values, dtype=tf.float32)

# 4. Convert labels to tensor (bool)
training_labels = tf.convert_to_tensor(training_labels.values, dtype=tf.bool)

# 5. Split normal vs anomalous
normal_training_data = tf.boolean_mask(training_data, ~training_labels)
anomalous_training_data = tf.boolean_mask(training_data, training_labels)

In [4]:
transformed_testing_data = pd.read_csv('Dataset/Transformed/Normalised/labelled_transformed_testing_data.csv')

# 1. Extract labels (column 6)
testing_labels = transformed_testing_data.iloc[:, 6].astype(bool)  # Anomalies: True

# 2. Drop label and extra column (column 6 and 7)
testing_data = transformed_testing_data.drop(
    columns=[transformed_testing_data.columns[6], transformed_testing_data.columns[7]]
)

# 3. Convert to float32 tensor
testing_data = tf.convert_to_tensor(testing_data.values, dtype=tf.float32)

# 4. Convert labels to tensor (bool)
testing_labels = tf.convert_to_tensor(testing_labels.values, dtype=tf.bool)

# 5. Split normal vs anomalous
normal_testing_data = tf.boolean_mask(testing_data, ~testing_labels)
anomalous_testing_data = tf.boolean_mask(testing_data, testing_labels)

In [5]:
transformed_validation_data = pd.read_csv('Dataset/Transformed/Normalised/labelled_transformed_validation_data.csv')

# 1. Extract labels (column 6)
validation_labels = transformed_validation_data.iloc[:, 6].astype(bool)  # Anomalies: True

# 2. Drop label and extra column (column 6 and 7)
validation_data = transformed_validation_data.drop(
    columns=[transformed_validation_data.columns[6], transformed_validation_data.columns[7]]
)

# 3. Convert to float32 tensor
validation_data = tf.convert_to_tensor(validation_data.values, dtype=tf.float32)

# 4. Convert labels to tensor (bool)
validation_labels = tf.convert_to_tensor(validation_labels.values, dtype=tf.bool)

# 5. Split normal vs anomalous
normal_validation_data = tf.boolean_mask(validation_data, ~validation_labels)
anomalous_validation_data = tf.boolean_mask(validation_data, validation_labels)

In [6]:
reconstructions = cae_model.predict(training_data)
reconstructions_error = tf.keras.losses.mae(reconstructions, training_data)

# Calculate mean and std of the reconstruction errors
mean_error = np.mean(reconstructions_error)
std_error = np.std(reconstructions_error)

# Set threshold to be mean + N * std, where N is a hyperparameter you can tune
threshold = mean_error + 2 * std_error  # For a more aggressive threshold, decrease the multiplier

print("Threshold: ", threshold)

# Make predictions based on the threshold
predictions = reconstructions_error > threshold

23849/23849 ━━━━━━━━━━━━━━━━━━━━ 10s 407us/step
Threshold:  0.023989638


In [7]:
import time

def predict(model, data, threshold):
    start_time = time.time()
    reconstructions = model(data)
    loss = tf.keras.losses.mae(reconstructions, data)
    # Detects if value greater than threshold then return true (for suspicious) else false (not suspicious)
    preds = tf.math.greater(loss, threshold)
    end_time = time.time()
    inference_time = end_time - start_time
    return loss.numpy(), preds.numpy().astype(int), inference_time

Do our Adversarial Optimisation

In [22]:
loss_fn = tf.keras.losses.MeanAbsoluteError()
x_start = tf.Variable(anomalous_testing_data[0:1])  
a_optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001)

n_steps = 10000  # To be tuned

for step in range(n_steps):
    with tf.GradientTape() as tape:
        x_adv = x_start  # Treat this as the adversarial candidate
        x_org = anomalous_testing_data[0:1]
        reconstruction = cae_model(x_adv)
        loss = loss_fn(x_adv, reconstruction)

        # Optionally: Add penalty term to keep x_adv close to original anomaly
        delta = tf.reduce_mean(tf.abs(x_adv - x_org), axis=1)
        total_loss = loss * delta  # λ = 0.01 to enforce anomaly preservation

    gradients = tape.gradient(total_loss, [x_start])
    a_optimizer.apply_gradients(zip(gradients, [x_start]))

    # Optional: clip to valid feature range if needed
    x_start.assign(tf.clip_by_value(x_start, clip_value_min=0.0, clip_value_max=1.0))

    if step % 50 == 0:
        print(f"Step {step}: Loss = {loss.numpy():.4f}, Delta = {delta.numpy().item():.4f}")


Step 0: Loss = 0.0534, Delta = 0.0000
Step 50: Loss = 0.0548, Delta = 0.1255
Step 100: Loss = 0.0538, Delta = 0.1263
Step 150: Loss = 0.0529, Delta = 0.1272
Step 200: Loss = 0.0521, Delta = 0.1280
Step 250: Loss = 0.0515, Delta = 0.1285
Step 300: Loss = 0.0509, Delta = 0.1292
Step 350: Loss = 0.0502, Delta = 0.1299
Step 400: Loss = 0.0495, Delta = 0.1307
Step 450: Loss = 0.0489, Delta = 0.1314
Step 500: Loss = 0.0486, Delta = 0.1315
Step 550: Loss = 0.0484, Delta = 0.1316
Step 600: Loss = 0.0481, Delta = 0.1318
Step 650: Loss = 0.0479, Delta = 0.1319
Step 700: Loss = 0.0476, Delta = 0.1321
Step 750: Loss = 0.0474, Delta = 0.1322
Step 800: Loss = 0.0471, Delta = 0.1324
Step 850: Loss = 0.0469, Delta = 0.1325
Step 900: Loss = 0.0466, Delta = 0.1327
Step 950: Loss = 0.0464, Delta = 0.1328
Step 1000: Loss = 0.0461, Delta = 0.1330
Step 1050: Loss = 0.0458, Delta = 0.1332
Step 1100: Loss = 0.0456, Delta = 0.1333
Step 1150: Loss = 0.0453, Delta = 0.1335
Step 1200: Loss = 0.0451, Delta = 0.133

In [28]:
# Get predictions
scores, y_pred, inference_time = predict(cae_model, x_start.numpy(), threshold=threshold)

print(f"Anomaly Score: {scores[0]:.4f}")
print("Prediction:", "Normal" if y_pred[0] == 0 else "Anomalous")


Anomaly Score: 0.0004
Prediction: Normal
